# Class 3 lab: can I trust this model?

**CML3113, Mining and Petroleum Engineering**  |  Week 10  |  KLAi at IUEA

Today you test how far you can trust a model that predicts **crusher energy demand** (in **kWh/t**): you watch a decision tree overfit, score it with cross-validation, plant three kinds of data leakage on purpose, and fix the random seed.

**How to use this notebook**

1. Run the cells in order from the top. Press Shift and Enter to run a cell.
2. Every step has the same rhythm: **PREDICT** what will happen, **RUN** the cell, **READ** the output, then **CHANGE** one thing and see what moves.
3. Type your answers in the lines that start with `>`. Double-click a text cell to edit it.
4. If you get an error, read the last line first. The slide called "Four problems you will meet" lists the common ones.

**The words** (also on the slides): *overfitting* means the model fits the noise in the training set, so it does well there and worse on new rows; *underfitting* means it is too simple to learn the pattern. The *bias-variance trade-off* is the balance between the two: a too-simple model has high bias, a too-flexible model has high variance. *Cross-validation* scores a model K times. The rows are cut into K parts called *folds*, and each fold takes one turn as the *validation fold* (also called the held-out fold; it is not the sealed test set E6). The *generalisation gap* is the validation error minus the training error. *Data leakage* is information from outside the training set reaching the model. A *random seed* (random_state in scikit-learn) makes a random choice repeat exactly. An *imputer* fills missing values (we use the median of the column). A *scaler* (StandardScaler) rescales columns to a common scale. Both learn numbers from the rows, so they must be fitted on the training set only.

The test set (group E6) stays sealed today. Nothing in this notebook uses it.

---
## Step 1: Prepare the data again

Same cleaning and the same split as Class 1, in a few lines. Upload `engineering_data.csv` in the Files panel if you have not already.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw_data = pd.read_csv("engineering_data.csv")
modelling_data = raw_data.drop_duplicates("record_id").copy()          # keep the first copy of each record_id
has_target = modelling_data["target"].notna()                          # a row needs a target
load_ok = modelling_data["load_pct"].between(0, 100)                   # load must be 0 to 100 percent
modelling_data = modelling_data[has_target & load_ok].copy()
print("rows left to model with:", len(modelling_data))

# split by equipment group, as in Class 1
training_data = modelling_data[modelling_data["equipment_id"].isin(["E1", "E2", "E3", "E4"])].copy()
validation_data = modelling_data[modelling_data["equipment_id"] == "E5"].copy()
test_data = modelling_data[modelling_data["equipment_id"] == "E6"].copy()      # sealed: we never use it today
print("training, validation, test rows:", len(training_data), len(validation_data), len(test_data))

feature_columns = ["load_pct", "ambient_c"]
X_train = training_data[feature_columns]
y_train = training_data["target"]
X_valid = validation_data[feature_columns]
y_valid = validation_data["target"]
groups = training_data["equipment_id"]        # which equipment group each training row belongs to

**READ the output.** Do the row counts match the "Check your numbers" slide? If not, stop and find the cell that went wrong.

> My answer: 

---
## Step 2: Underfitting, a good fit and overfitting: the validation curve

We train decision trees with max_depth 1 to 10 (max_depth is a hyperparameter: a setting chosen before training) and compare the error on the training set with the error on the validation set, rows the trees have never seen. Plotting both errors against max_depth gives a validation curve.

**PREDICT first.** As the depth grows from 1 to 10, what will happen to the training error? And to the validation error?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

depths = list(range(1, 11))
train_maes = []
valid_maes = []
for depth in depths:
    tree_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=depth, random_state=17))
    tree_model.fit(X_train, y_train)                                      # training the model
    train_mae = mean_absolute_error(y_train, tree_model.predict(X_train))     # training set
    valid_mae = mean_absolute_error(y_valid, tree_model.predict(X_valid))     # validation set
    train_maes.append(train_mae)
    valid_maes.append(valid_mae)
    print(f"depth {depth:2d}  training {train_mae:.2f}  validation {valid_mae:.2f}  generalisation gap {valid_mae - train_mae:.2f}")

**READ the output.** Where does the training error reach zero? Where does the validation error stop improving? How large is the generalisation gap at depth 10? Which depths look like underfitting and which like overfitting?

> My answer: 

**Choose max_depth with the validation set only.** The validation set has only a few rows, so its scores are noisy. Our rule is model selection by parsimony (prefer the simpler model when scores are close): find the lowest validation error, accept any depth within a tolerance of 5 percent of it, and take the simplest (smallest) depth among them. This is a common practical rule, not a law, and it is similar in spirit to the one-standard-error rule.

In [ ]:
lowest = min(valid_maes)
within_tolerance = [d for d, v in zip(depths, valid_maes) if v <= lowest * 1.05]   # depths within 5 percent of the lowest
best_depth = min(within_tolerance)                                                  # the simplest of them
print(f"lowest validation MAE: {lowest:.2f} kWh/t")
print("depths within the tolerance:", within_tolerance)
print("chosen depth:", best_depth)

plt.figure(figsize=(6, 3.8))
plt.plot(depths, train_maes, marker="o", label="training")
plt.plot(depths, valid_maes, marker="o", label="validation")
plt.axvline(best_depth, linestyle="--")
plt.xlabel("max_depth")
plt.ylabel("MAE (kWh/t)")
plt.legend()
plt.title("Validation curve: training and validation error")
plt.show()

**READ the output.** Is the chosen depth the one with the very lowest validation error? Why did we accept the tolerance? Did we use the test group E6 anywhere?

> My answer: 

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = make_pipeline(SimpleImputer(strategy="median"), LinearRegression())
linear_model.fit(X_train, y_train)
linear_mae = mean_absolute_error(y_valid, linear_model.predict(X_valid))
print(f"Linear regression, validation MAE: {linear_mae:.2f} kWh/t")

**READ the output.** Compare the linear regression model from Class 1 with every tree in the table above. What does that say about the shape of this data?

> My answer: 

**CHANGE one thing.** Change `lowest * 1.05` to `lowest * 1.00` (no tolerance) and run the cell again. Which depth is chosen now? Then put it back to 1.05. Remember the depths 1, 2 and 8 you tried at the end of Class 1: this curve is what you were seeing.

> What I saw: 

---
## Step 3: One split can be lucky: K-fold and GroupKFold cross-validation

We score the max_depth 4 tree from Class 1 on the training rows only. K-fold cross-validation cuts the rows into folds. Each fold takes one turn as the validation fold (held-out fold) while the tree is trained on the other folds. We do it with a loop first, then with the one-line shortcut cross_val_score. Each score is a MAE, and we report their mean and standard deviation.

**PREDICT first.** Five fold scores will not all be equal. Will they differ by a little or by a lot? Will their mean be close to the single validation score of the depth-4 tree?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

kfold = KFold(n_splits=5)          # cut the 58 training rows into 5 folds, in order
cv_scores = []
for number, (train_idx, valid_idx) in enumerate(kfold.split(X_train), start=1):
    tree_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=17))
    tree_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])           # the imputer is also fitted on the training folds only
    y_pred = tree_model.predict(X_train.iloc[valid_idx])
    mae = mean_absolute_error(y_train.iloc[valid_idx], y_pred)
    cv_scores.append(mae)
    print(f"fold {number}: train on {len(train_idx)} rows, validation fold has {len(valid_idx)} rows, MAE {mae:.2f}")

print(f"5-fold CV mean MAE: {np.mean(cv_scores):.2f}   standard deviation: {np.std(cv_scores):.2f}")
print(f"lowest to highest: {min(cv_scores):.2f} to {max(cv_scores):.2f}")

**READ the output.** Pick one fold and check that its validation rows were not among its training rows (the two counts add up to 58). Is the standard deviation small or large compared with the mean?

> My answer: 

In [ ]:
# the one-line shortcut, cross_val_score, gives the same numbers
imputer = SimpleImputer(strategy="median")
tree = DecisionTreeRegressor(max_depth=4, random_state=17)
pipe = make_pipeline(imputer, tree)
cv_scores_auto = -cross_val_score(pipe, X_train, y_train, cv=KFold(n_splits=5), scoring="neg_mean_absolute_error")   # minus: scikit-learn flips the sign
print(f"cross_val_score mean MAE: {cv_scores_auto.mean():.2f}")
print("same scores as the loop:", bool(np.allclose(cv_scores_auto, cv_scores)))

**GroupKFold.** Each fold holds out one whole equipment group as the validation fold. The model is tested on a group it has never met, which is how it will be used on a new machine.

In [ ]:
group_kfold = GroupKFold(n_splits=4)
results = {}
for train_idx, valid_idx in group_kfold.split(X_train, y_train, groups):
    held_out = groups.iloc[valid_idx].iloc[0]                  # the name of the held-out group
    tree_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=17))
    tree_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
    results[held_out] = mean_absolute_error(y_train.iloc[valid_idx], tree_model.predict(X_train.iloc[valid_idx]))

for name in sorted(results):
    print(f"held-out group {name}: MAE {results[name]:.2f}")
group_values = list(results.values())
print(f"GroupKFold 4-fold CV mean MAE: {np.mean(group_values):.2f}   standard deviation: {np.std(group_values):.2f}")

cv_scores_group = -cross_val_score(pipe, X_train, y_train, cv=group_kfold, groups=groups, scoring="neg_mean_absolute_error")
print(f"cross_val_score mean MAE: {cv_scores_group.mean():.2f}")

# the single split from Class 1, for comparison
single_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=17)).fit(X_train, y_train)
print(f"one split, group E5: MAE {mean_absolute_error(y_valid, single_model.predict(X_valid)):.2f} kWh/t")

**READ the output.** Which held-out group is hardest? How does the one-split score for E5 compare with the GroupKFold mean? What does the standard deviation tell you about how far to trust one number?

> My answer: 

**CHANGE one thing.** Change `max_depth=4` to your chosen depth from Step 2 in the plain 5-fold loop (rows cut in order, no shuffle) and run it again. Does the mean error get smaller or larger? Does the standard deviation change?

> What I saw: 

---
## Step 4: Plant three kinds of data leakage on purpose

Data leakage makes the score look better than it really is. We plant each of the three kinds from the slides and read the damage.

**Leakage 1, train-test contamination: the validation rows are copied into the training set.** We use the max_depth 10 tree so that overfitting is easy.

**PREDICT first.** The leaky tree was trained on the validation rows. What validation error do you expect from it?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
honest_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=10, random_state=17))
honest_model.fit(X_train, y_train)
honest_mae = mean_absolute_error(y_valid, honest_model.predict(X_valid))

X_leaky = pd.concat([X_train, X_valid])         # the mistake: validation rows join the training set
y_leaky = pd.concat([y_train, y_valid])
leaky_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=10, random_state=17))
leaky_model.fit(X_leaky, y_leaky)
leaky_mae = mean_absolute_error(y_valid, leaky_model.predict(X_valid))

print(f"Honest validation MAE: {honest_mae:.2f} kWh/t")
print(f"Leaky validation MAE: {leaky_mae:.2f} kWh/t")

**READ the output.** Which score would you quote to a client? Which score would predict a brand new machine?

> My answer: 

**Leakage 2, preprocessing leakage: the imputer is fitted on all rows.** One missing value (ambient_c in one training row) is filled by median imputation. The honest fill uses the median of the training rows. The leaky fill uses the median of training and validation rows together.

In [ ]:
X_all = pd.concat([X_train, X_valid])
fill_train = X_train.median()          # median of each column, from the training rows only
fill_all = X_all.median()              # the mistake: validation rows help decide the fill
print(f"fill for ambient_c, training rows only: {fill_train['ambient_c']:.2f}")
print(f"fill for ambient_c, all rows (leaky): {fill_all['ambient_c']:.2f}")

tree_train_fill = DecisionTreeRegressor(max_depth=10, random_state=17).fit(X_train.fillna(fill_train), y_train)
tree_all_fill = DecisionTreeRegressor(max_depth=10, random_state=17).fit(X_train.fillna(fill_all), y_train)
print(f"MAE with the honest fill: {mean_absolute_error(y_valid, tree_train_fill.predict(X_valid.fillna(fill_train))):.2f} kWh/t")
print(f"MAE with the leaky fill: {mean_absolute_error(y_valid, tree_all_fill.predict(X_valid.fillna(fill_all))):.2f} kWh/t")

**READ the output.** The two fill values differ, but the two scores are the same. Does that mean the leaky fill is safe? Why? (Hint: how many values are missing here?) Which tool fits the imputer on the training folds only, in every fold? (Hint: it is the Pipeline.)

> My answer: 

**Leakage 3, target leakage: a feature built from the target.** We add the target rounded to a whole number as a feature.

In [ ]:
X_train_leak = X_train.copy()
X_valid_leak = X_valid.copy()
X_train_leak["target_rounded"] = y_train.round()       # the mistake: built from the target
X_valid_leak["target_rounded"] = y_valid.round()
leak_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=17))
leak_model.fit(X_train_leak, y_train)
print(f"Depth-4 tree with the target feature: {mean_absolute_error(y_valid, leak_model.predict(X_valid_leak)):.2f} kWh/t")
clean_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=17)).fit(X_train, y_train)
print(f"Depth-4 tree, honest features: {mean_absolute_error(y_valid, clean_model.predict(X_valid)):.2f} kWh/t")

**READ the output.** The first score looks wonderful. Why could you never use that feature when a new reading arrives?

> My answer: 

**THINK.** Write down which of the five checklist questions on the slide each of the three kinds of leakage breaks.

> My answer: 

---
## Step 5: Reproducibility: the random seed and a run record

The random seed (random_state in scikit-learn) makes a random choice repeat exactly. The same seed gives the same numbers. A different seed can give a different number.

**PREDICT first.** If we change only the random seed, will the max_depth 10 tree score change? Will the shuffled 5-fold average change?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
for seed in [1, 17, 99]:
    tree_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=10, random_state=seed))
    tree_model.fit(X_train, y_train)
    print(f"tree random_state {seed}: validation MAE {mean_absolute_error(y_valid, tree_model.predict(X_valid)):.2f} kWh/t")

pipe = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=17))
for seed in [1, 17, 99]:
    kfold_shuffled = KFold(n_splits=5, shuffle=True, random_state=seed)          # shuffle the rows before cutting them into folds
    cv_scores = -cross_val_score(pipe, X_train, y_train, cv=kfold_shuffled, scoring="neg_mean_absolute_error")
    print(f"shuffled 5-fold, random_state {seed}: mean MAE {cv_scores.mean():.2f}")

**READ the output.** Which numbers moved? By how much? Why must a result always be quoted together with its random seed? What does the range of the three numbers tell you about luck?

> My answer: 

In [ ]:
import platform
import hashlib
import sklearn

fingerprint = hashlib.md5(open("engineering_data.csv", "rb").read()).hexdigest()[:8]
print("RUN RECORD")
print("data file: engineering_data.csv,", len(raw_data), "rows, fingerprint", fingerprint)
print("rows after cleaning in Class 1:", len(modelling_data), "(training", len(training_data), ", validation", len(validation_data), ", test", len(test_data), ")")
print("split: training E1 to E4, validation E5, test E6 sealed")
print("random seed: random_state = 17 for trees and for the shuffled folds (Step 5 also tries 1 and 99)")
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

Copy this record under every result you report. Your version numbers may differ from your neighbour's.

---
## Step 6: Your conclusion, and save your work

Write three sentences: which max_depth you chose and why, what cross-validation says about how far to trust the score, and which kind of data leakage surprised you most.

> My conclusion: 

**What this does not prove.** The data are synthetic teaching data, the validation group has only a few rows, and the test set (group E6) has not been used. Choose **Runtime, then Restart session and run all**. If every cell runs without an error, choose **File, then Save a copy in Drive**. Keep the notebook: next class we extend it.